# Macrocosm photo-z CNN — train your own version (Colab + MLflow)

**One self-contained notebook**, and it **streams the data — only one batch is ever in RAM**
(the image shards are memory-mapped on local disk, rows are read on the fly). So you can train on
the full 600k at 64×64 without a High-RAM runtime; RAM is no longer the limit — your local disk
(how many shards you download) is.

**To make your own version:** edit the **MODEL cell (section 2)**, name your run in the MLflow cell,
run all. **Runtime → Change runtime type → GPU** first.

## 0. Setup (Colab) — clone repo, install, log in, download all data
Clones **CNN-Model** (brings `eval.py` + `splits/` into Colab so you can score on the fixed 50k val),
installs deps, and downloads **all** image shards (~21 GB to local disk — streaming keeps RAM
untouched). Takes a couple of minutes.

In [ ]:
# clone the repo (eval.py + splits/) and cd into it
![ -d /content/CNN-Model ] || git clone https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /content/CNN-Model
%cd /content/CNN-Model
!pip -q install mlflow
# Google login (project member) for GCS
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/
# download ALL image shards (~21 GB; 91/100 present — shards 65-73 not built yet)
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_*.npy /content/data/
DATA_DIR = '/content/data'

## 1. Pick the training data — a SUBSET of the canonical train split
Train only on objids from `splits/train_objids.csv` (**never** the held-out 50k val — that's what
keeps `eval.py` honest). `is_train_subset` checks your csv doesn't leak. Default = all 550k; point
`TRAIN_CSV` at your own subset csv to train on less. Only row indices enter RAM here, not images.

In [ ]:
import glob, re, numpy as np, pandas as pd
from eval import is_train_subset
SHARD = 6000
DATA_DIR = globals().get('DATA_DIR', '/content/data')

# your training objids -- MUST be a subset of splits/train_objids.csv (no 50k-val leakage)
TRAIN_CSV = globals().get('TRAIN_CSV', 'splits/train_objids.csv')   # or your own subset csv
chk = is_train_subset(TRAIN_CSV)
assert chk['ok'], f"LEAK: {chk['n_outside_train']} objids in {TRAIN_CSV} are NOT in the train split {chk['sample_outside']}"
print(f"train csv OK: {chk['n']:,} objids, all within the canonical train split")

cat = pd.read_parquet(f'{DATA_DIR}/catalog_v1.parquet', columns=['objid', 'redshift'])
z_all = cat['redshift'].values
o2i = {int(o): i for i, o in enumerate(cat['objid'].values)}
idx = np.array([o2i[int(o)] for o in pd.read_csv(TRAIN_CSV)['objid'].values], dtype=np.int64)
# keep only rows whose image shard is actually downloaded
present = set(int(re.findall(r'images_(\d+)_', p)[0]) // SHARD for p in glob.glob(f'{DATA_DIR}/images_*.npy'))
idx = idx[[(i // SHARD) in present for i in idx]]
N = globals().get('N', len(idx))                      # optionally cap for a quick run
rng = np.random.RandomState(0); rng.shuffle(idx); idx = idx[:N]
# small early-stopping set, held out FROM TRAIN (NOT the canonical 50k val)
es_idx, train_idx = idx[:2000], idx[2000:]
zes = z_all[es_idx]
print(f'train {len(train_idx):,} / early-stop {len(es_idx):,}   (canonical 50k val held by eval.py)')

## 2. 👉 THE MODEL — **edit this cell** to redesign the architecture
Example: VGG stem (cheap 64→16 downsample) + 3 lightweight **Inception** modules → GAP →
a 64-d **`embedding`** (feeds the fusion head later) → `z`. ~288K params. Change anything:
more/fewer Inception blocks, deeper stem, a classification head, transfer learning. (KB MCM-A-18)

In [ ]:
from tensorflow.keras import layers as L, Model, Input

def inception(x, f1, f3r, f3, f5r, f5, fp, name):
    """4 parallel branches concatenated on channels; 1x1 bottlenecks keep it cheap."""
    b1 = L.Conv2D(f1, 1, padding='same', activation='relu', name=f'{name}_1x1')(x)
    b3 = L.Conv2D(f3r, 1, padding='same', activation='relu', name=f'{name}_3x3_reduce')(x)
    b3 = L.Conv2D(f3, 3, padding='same', activation='relu', name=f'{name}_3x3')(b3)
    b5 = L.Conv2D(f5r, 1, padding='same', activation='relu', name=f'{name}_5x5_reduce')(x)
    b5 = L.Conv2D(f5, 5, padding='same', activation='relu', name=f'{name}_5x5')(b5)
    bp = L.MaxPool2D(3, strides=1, padding='same', name=f'{name}_pool')(x)
    bp = L.Conv2D(fp, 1, padding='same', activation='relu', name=f'{name}_pool_proj')(bp)
    return L.Concatenate(axis=-1, name=f'{name}_concat')([b1, b3, b5, bp])

def build_cnn(input_shape=(64, 64, 5), embed_dim=64):
    inp = Input(shape=input_shape, name='cutout')
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1a')(inp)
    x = L.Conv2D(32, 3, padding='same', activation='relu', name='stem1b')(x)
    x = L.BatchNormalization(name='stem1_bn')(x); x = L.MaxPool2D(name='stem1_pool')(x)   # 32x32
    x = L.Conv2D(64, 3, padding='same', activation='relu', name='stem2')(x)
    x = L.BatchNormalization(name='stem2_bn')(x); x = L.MaxPool2D(name='stem2_pool')(x)   # 16x16
    x = inception(x, 32, 32, 48, 8, 24, 24, name='inc1'); x = L.BatchNormalization(name='inc1_bn')(x)
    x = L.MaxPool2D(name='inc1_down')(x)                                                  # 8x8
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc2'); x = L.BatchNormalization(name='inc2_bn')(x)
    x = L.MaxPool2D(name='inc2_down')(x)                                                  # 4x4
    x = inception(x, 64, 48, 96, 16, 48, 48, name='inc3'); x = L.BatchNormalization(name='inc3_bn')(x)
    x = L.GlobalAveragePooling2D(name='gap')(x)
    x = L.Dense(128, activation='relu', name='dense')(x); x = L.Dropout(0.3, name='dropout')(x)
    emb = L.Dense(embed_dim, activation='relu', name='embedding')(x)
    zout = L.Dense(1, name='z')(emb)
    return Model(inp, zout, name='photoz_cnn')

def build_embedder(cnn):
    return Model(cnn.input, cnn.get_layer('embedding').output, name='photoz_embedder')

model = build_cnn()
model.summary()
print('params:', f'{model.count_params():,}')

## 3. Streaming pipeline + metrics — usually leave as-is
`streaming_dataset` memory-maps the shards and yields cropped rows on the fly, so **only the
current batch is in RAM**. Same arcsinh + per-image norm as the serving backend (no skew).

In [ ]:
import tensorflow as tf

def preprocess(x, y):                      # x: (B,H,W,5) batched -> arcsinh + per-image per-channel norm
    x = tf.cast(x, tf.float32); x = tf.math.asinh(x)
    m = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
    s = tf.math.reduce_std(x, axis=[1, 2], keepdims=True) + 1e-6
    return (x - m) / s, y

def augment(x, y):                         # one random 90-deg rot + flips per batch
    x = tf.image.rot90(x, tf.random.uniform([], 0, 4, dtype=tf.int32))
    x = tf.image.random_flip_left_right(x); x = tf.image.random_flip_up_down(x)
    return x, y

def streaming_dataset(idx, training=False, batch=128, crop=64):
    """ONE numpy read per BATCH (vectorized per-shard), parallel across batches -> GPU well-fed.
    Only batches enter RAM (shuffle is on indices). Gaps OK (dict keyed by shard number)."""
    paths = glob.glob(f'{DATA_DIR}/images_*.npy')
    mm = {int(re.findall(r'images_(\d+)_', p)[0]) // SHARD: np.load(p, mmap_mode='r')
          for p in paths}                               # key = SHARD NUMBER (gaps OK), lazy memmaps
    o = (64 - crop) // 2
    def read_batch(idxb):                               # idxb (B,) -> (B,crop,crop,5), (B,)
        idxb = idxb.astype(np.int64)
        out = np.empty((len(idxb), crop, crop, 5), np.float32)
        for s in np.unique(idxb // SHARD):              # one vectorised read per shard touched
            sel = np.where(idxb // SHARD == s)[0]; rows = idxb[sel] % SHARD
            out[sel] = mm[int(s)][rows][:, o:o+crop, o:o+crop, :]
        return out, np.log1p(z_all[idxb]).astype('float32')
    def tf_read(idxb):                                  # one map, one arg -> no unpacking surprises
        x, y = tf.numpy_function(read_batch, [idxb], [tf.float32, tf.float32])
        x.set_shape((None, crop, crop, 5)); y.set_shape((None,))
        x, y = preprocess(x, y)                         # call them directly (not chained .map)
        if training:
            x, y = augment(x, y)
        return x, y
    ds = tf.data.Dataset.from_tensor_slices(np.asarray(idx, np.int64))
    if training:
        ds = ds.shuffle(min(len(idx), 100_000), reshuffle_each_iteration=True).repeat()  # shuffle INDICES
    return ds.batch(batch).map(tf_read, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def _dz(zt, zp): zt, zp = np.asarray(zt), np.asarray(zp); return (zp - zt) / (1 + zt)
def sigma_mad(zt, zp): d = _dz(zt, zp); return float(1.4826 * np.median(np.abs(d - np.median(d))))
def outlier_rate(zt, zp): return float(np.mean(np.abs(_dz(zt, zp)) > 0.05))

class SigmaMadCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, z_true): super().__init__(); self.val_ds = val_ds; self.z_true = np.asarray(z_true)
    def on_epoch_end(self, epoch, logs=None):
        zp = np.expm1(self.model.predict(self.val_ds, verbose=0).ravel())
        sm, out = sigma_mad(self.z_true, zp), outlier_rate(self.z_true, zp)
        logs = logs if logs is not None else {}
        logs['val_sigma_MAD'] = sm; logs['val_outlier'] = out
        try:                       # log metrics OURSELVES each epoch -- autolog misses them on Keras 3
            import mlflow
            if mlflow.active_run():
                mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=epoch)
        except Exception as e:
            print('  (mlflow metric log skipped:', e, ')')
        print(f'  -> val sigma_MAD={sm:.4f}  outlier={out*100:.1f}%')

## 4. Connect MLflow (optional)
Paste the bearer token (ask the team — not in git); start the server first (`make mlflow-start`).
Without a token it just trains and skips logging.

In [ ]:
import os
MLFLOW_TOKEN = 'PASTE_MLFLOW_API_TOKEN_HERE'
USE_MLFLOW = 'PASTE' not in MLFLOW_TOKEN
if USE_MLFLOW:
    import mlflow, mlflow.tensorflow
    os.environ['MLFLOW_TRACKING_URI'] = 'https://146-148-10-86.sslip.io'
    os.environ['MLFLOW_TRACKING_TOKEN'] = MLFLOW_TOKEN
    mlflow.set_experiment('photoz-cnn')
    mlflow.tensorflow.autolog(log_models=True)
    print('MLflow: logging to', os.environ['MLFLOW_TRACKING_URI'])
else:
    print('MLflow token not set -> training without logging (fine for a quick test)')

## 5. Train + log
Trains on **your train subset**, early-stops on a small held-out-**from-train** set, then scores the
**canonical 50k val** via `eval.py` — that's the number we compare (`val50k_sigma_MAD`). Name **your**
run. Logs the full **recipe** to MLflow: a `CONFIG` (incl. `train_csv`) + a `recipe.py` artifact with
the exact source of `preprocess`/`augment`/`streaming_dataset`/`build_cnn` — fully reproducible.

In [ ]:
EPOCHS = globals().get('EPOCHS', 50)
RUN_NAME = 'example_vgg_inception'      # <- name YOUR run

import inspect
from eval import evaluate
CROP, BATCH, LR = 64, 128, 1e-3            # config vars -> logged (not hard-coded in the log)
train_ds = streaming_dataset(train_idx, training=True, batch=BATCH, crop=CROP)
es_ds    = streaming_dataset(es_idx,    training=False, batch=256, crop=CROP)
steps    = len(train_idx) // BATCH                  # repeated train stream -> set the cadence
es_steps = int(np.ceil(len(es_idx) / 256))
model.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss=tf.keras.losses.Huber(delta=0.02), metrics=['mae'])
cbs = [SigmaMadCallback(es_ds, zes),               # early-stop signal (held out FROM train)
       tf.keras.callbacks.EarlyStopping('val_loss', patience=6, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3, min_lr=1e-5)]

# everything that defines this run -> logged to MLflow (table) + recipe.py artifact (exact code)
CONFIG = dict(crop=CROP, batch=BATCH, lr=LR, optimizer='adam', loss='huber(0.02)', target='log1p(z)',
              stretch='arcsinh', norm='per-image per-channel', augment='rot90+flip',
              arch='vgg-stem+3inception', train_csv=TRAIN_CSV, N=len(train_idx),
              params=int(model.count_params()))

def run_training():
    model.fit(train_ds, validation_data=es_ds, epochs=EPOCHS, steps_per_epoch=steps,
              validation_steps=es_steps, callbacks=cbs)
    m = evaluate(model, data_dir=DATA_DIR, crop=CROP)   # CANONICAL score on the fixed 50k val
    print('50k val:', m)
    return m

if USE_MLFLOW:
    import mlflow
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params(CONFIG)                                  # preprocessing + training config
        mlflow.log_text('\n\n'.join(inspect.getsource(f)         # the exact preprocessing/model code
                        for f in (preprocess, augment, streaming_dataset, build_cnn)), 'recipe.py')
        m = run_training()
        mlflow.log_metrics({'val50k_sigma_MAD': m['sigma_MAD'], 'val50k_outlier': m['outlier'], 'val50k_n': m['n']})
else:
    run_training()

## 6. Compare
Open the MLflow UI (`https://146-148-10-86.sslip.io`, GitHub-org login) and sort runs by
**`val50k_sigma_MAD`** — everyone scored on the *same* fixed 50k val, so it's directly comparable.
The lowest one wins (its `embedding` feeds the fusion head).